# Lab: Support Vector Machine (SVM)

## 1. Ý tưởng cốt lõi

Cho hai lớp điểm trong không gian, có vô số đường (siêu phẳng) tách được chúng. SVM hỏi: **đường nào là tốt nhất?**

Câu trả lời: đường tách *xa nhất* khỏi cả hai lớp. "Khoảng cách từ đường tách đến điểm gần nhất của mỗi lớp" gọi là **margin**. SVM tìm siêu phẳng có **margin lớn nhất** — gọi là **maximum margin classifier**.

Trực giác: margin lớn = an toàn hơn khi gặp dữ liệu mới hơi lệch một chút.

## 2. Hard-margin SVM (dữ liệu tách tuyến tính)

Giả sử nhãn $y_i \in \{-1, +1\}$. Siêu phẳng là $w^T x + b = 0$. Ta muốn:
$$
y_i (w^T x_i + b) \ge 1, \quad \forall i
$$
Margin $= \frac{2}{\|w\|}$ → tối đa hoá margin = **tối thiểu** $\frac{1}{2}\|w\|^2$.

**Bài toán tối ưu (primal):**
$$
\min_{w, b} \frac{1}{2}\|w\|^2 \quad \text{s.t.} \quad y_i(w^T x_i + b) \ge 1
$$

## 3. Soft-margin SVM (dữ liệu không tách hoàn toàn)

Thực tế dữ liệu có nhiễu, có outlier. Cho phép một số điểm vi phạm margin bằng biến slack $\xi_i \ge 0$:
$$
y_i(w^T x_i + b) \ge 1 - \xi_i
$$

$$
\min_{w, b, \xi} \frac{1}{2}\|w\|^2 + C\sum_i \xi_i
$$

**Tham số $C$**:
- $C$ lớn → phạt vi phạm nặng → margin nhỏ, sát dữ liệu (dễ overfit).
- $C$ nhỏ → khoan dung hơn → margin lớn (bias cao nhưng generalize tốt).

## 4. Bài toán đối ngẫu (dual) và kernel

Bằng nhân tử Lagrange, primal có thể chuyển sang **dual**:
$$
\max_{\lambda} \sum_i \lambda_i - \frac{1}{2}\sum_{i, j}\lambda_i \lambda_j y_i y_j (x_i \cdot x_j)
$$
$$
\text{s.t.} \quad 0 \le \lambda_i \le C, \quad \sum_i \lambda_i y_i = 0
$$

Điểm quan trọng: dual chỉ phụ thuộc vào *tích vô hướng* $x_i \cdot x_j$. Đây là cánh cửa đến **Kernel Trick** — thay $x_i \cdot x_j$ bằng một hàm $K(x_i, x_j)$ tương ứng với tích vô hướng trong một không gian cao chiều ngầm. SVM lúc đó học được ranh giới phi tuyến mà không cần thực sự nhảy lên không gian cao.

## 5. Các kernel phổ biến

- **Linear**: $K(x, y) = x \cdot y$.
- **Polynomial**: $K(x, y) = (x \cdot y + c)^d$.
- **RBF (Gaussian)**: $K(x, y) = \exp(-\gamma\|x - y\|^2)$. Phổ biến nhất, hoạt động tốt cho hầu hết dataset.

## 6. Support vectors

Sau khi train, chỉ những điểm có $\lambda_i > 0$ mới đóng góp vào quyết định. Đây là **support vectors** — thường chỉ một phần nhỏ của dữ liệu. Đó là lý do SVM gọi là *Support Vector* Machine.

# THỰC HÀNH 1: Linear SVM trên dữ liệu 2D giả lập

Sinh hai cụm có thể tách tuyến tính, train SVM, vẽ siêu phẳng + margin + support vectors.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs, make_moons
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report

np.random.seed(42)

In [ ]:
# Hai cụm tách tuyến tính
X, y = make_blobs(n_samples=80, centers=2, cluster_std=1.0, random_state=42)

clf = SVC(kernel='linear', C=1.0)
clf.fit(X, y)

def plot_svm(clf, X, y, title):
    plt.figure(figsize=(7, 6))
    plt.scatter(X[:, 0], X[:, 1], c=y, cmap='coolwarm', s=30, edgecolor='k')

    # Vẽ ranh giới + margin
    xx = np.linspace(X[:, 0].min() - 1, X[:, 0].max() + 1, 200)
    yy = np.linspace(X[:, 1].min() - 1, X[:, 1].max() + 1, 200)
    XX, YY = np.meshgrid(xx, yy)
    grid = np.c_[XX.ravel(), YY.ravel()]
    Z = clf.decision_function(grid).reshape(XX.shape)
    plt.contour(XX, YY, Z, levels=[-1, 0, 1], colors=['red', 'black', 'red'],
                linestyles=['--', '-', '--'])

    # Tô đậm support vectors
    plt.scatter(clf.support_vectors_[:, 0], clf.support_vectors_[:, 1],
                s=200, facecolors='none', edgecolors='lime', linewidths=2,
                label=f'{len(clf.support_vectors_)} support vectors')
    plt.legend(); plt.title(title); plt.grid(alpha=0.3)
    plt.show()

plot_svm(clf, X, y, f'Linear SVM, C={clf.C}')
print(f'w = {clf.coef_[0]},  b = {clf.intercept_[0]:.3f}')
print(f'Margin width = 2/||w|| = {2 / np.linalg.norm(clf.coef_):.3f}')

### Ảnh hưởng của tham số C

Thêm vài điểm "khó" gần boundary để thấy C ảnh hưởng thế nào.

In [ ]:
# Dữ liệu hơi bị overlap
X2, y2 = make_blobs(n_samples=80, centers=2, cluster_std=1.8, random_state=42)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, c in zip(axes, [0.01, 1, 100]):
    clf = SVC(kernel='linear', C=c).fit(X2, y2)
    ax.scatter(X2[:, 0], X2[:, 1], c=y2, cmap='coolwarm', s=30, edgecolor='k')
    xx = np.linspace(X2[:, 0].min()-1, X2[:, 0].max()+1, 100)
    yy = np.linspace(X2[:, 1].min()-1, X2[:, 1].max()+1, 100)
    XX, YY = np.meshgrid(xx, yy)
    Z = clf.decision_function(np.c_[XX.ravel(), YY.ravel()]).reshape(XX.shape)
    ax.contour(XX, YY, Z, levels=[-1, 0, 1], colors=['red', 'black', 'red'],
               linestyles=['--', '-', '--'])
    ax.scatter(clf.support_vectors_[:, 0], clf.support_vectors_[:, 1],
               s=150, facecolors='none', edgecolors='lime', linewidths=2)
    ax.set_title(f'C = {c}, {len(clf.support_vectors_)} SVs, margin = {2/np.linalg.norm(clf.coef_):.2f}')
    ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()
print('C nhỏ (0.01): margin lớn, nhiều SV — model "khoan dung" với vi phạm.')
print('C lớn (100): margin hẹp, ít SV — model cố gò sát dữ liệu, dễ overfit.')

# THỰC HÀNH 2: Kernel SVM trên dữ liệu phi tuyến

Dùng `make_moons` — hai trăng lưỡi liềm cài vào nhau, không tách được tuyến tính.

In [ ]:
X, y = make_moons(n_samples=200, noise=0.2, random_state=42)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, kernel in zip(axes, ['linear', 'poly', 'rbf']):
    clf = SVC(kernel=kernel, C=1.0, gamma='scale', degree=3).fit(X, y)
    ax.scatter(X[:, 0], X[:, 1], c=y, cmap='coolwarm', s=30, edgecolor='k')
    xx = np.linspace(X[:, 0].min()-0.5, X[:, 0].max()+0.5, 200)
    yy = np.linspace(X[:, 1].min()-0.5, X[:, 1].max()+0.5, 200)
    XX, YY = np.meshgrid(xx, yy)
    Z = clf.decision_function(np.c_[XX.ravel(), YY.ravel()]).reshape(XX.shape)
    ax.contourf(XX, YY, Z, levels=20, cmap='coolwarm', alpha=0.3)
    ax.contour(XX, YY, Z, levels=[0], colors='black', linewidths=2)
    ax.set_title(f'Kernel: {kernel}, train acc = {clf.score(X, y)*100:.1f}%')
    ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()
print('Kernel linear không vượt quá ~85% trên dữ liệu cong.')
print('Kernel RBF dễ dàng đạt >95% — học được ranh giới cong.')

## 7. SVM trên dataset thật (Iris)

In [ ]:
from sklearn.datasets import load_iris

data = load_iris()
X, y = data.data, data.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y)

# QUAN TRỌNG: SVM nhạy với scale → chuẩn hoá trước.
# Fit scaler chỉ trên train.
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

# Grid search tìm best C, gamma
param_grid = {'C': [0.1, 1, 10, 100], 'gamma': ['scale', 0.01, 0.1, 1]}
gs = GridSearchCV(SVC(kernel='rbf'), param_grid, cv=5, scoring='accuracy')
gs.fit(X_train_s, y_train)

print(f'Best params: {gs.best_params_}')
print(f'Best CV acc: {gs.best_score_*100:.2f}%')
print(f'Test acc:    {gs.score(X_test_s, y_test)*100:.2f}%')
print()
print(classification_report(y_test, gs.predict(X_test_s),
                            target_names=data.target_names))

## Tổng kết

1. SVM tìm siêu phẳng có **margin tối đa**.
2. **Tham số C**: kiểm soát đánh đổi giữa margin lớn (C nhỏ) và phạt vi phạm (C lớn).
3. **Kernel trick**: cho phép học ranh giới phi tuyến mà không cần explicitly nhảy lên không gian cao chiều.
4. RBF kernel là default tốt; chỉ cần tune $C$ và $\gamma$.
5. **PHẢI scale feature** — SVM nhạy với scale.
6. Số tham số SVM = số support vectors → mô hình "thưa" tự nhiên.

## Khi nào dùng SVM?
- Dataset vừa và nhỏ (vài nghìn đến vài chục nghìn).
- Dữ liệu có chiều cao nhưng vẫn ít hơn số mẫu.
- Cần ranh giới quyết định rõ ràng.

## Khi nào tránh SVM?
- Dataset cực lớn (>100k mẫu) — train chậm.
- Cần predict probability đáng tin (SVM không cho probability tự nhiên; `probability=True` chậm và không phải lúc nào cũng tốt).

# BÀI TẬP VỀ NHÀ

## Bài 1: Vẽ ảnh hưởng của gamma trong RBF
Trên `make_moons`, train SVM RBF với `gamma ∈ {0.1, 1, 10, 100}`, C cố định = 1. Vẽ 4 decision boundary. Quan sát:
- gamma nhỏ → boundary mượt (underfit).
- gamma lớn → boundary nhảy theo từng điểm (overfit).

## Bài 2: SVM trên drug200
Apply SVM lên drug200 (như bài Decision Tree). Sweep `kernel ∈ {linear, rbf, poly}`. So sánh với Random Forest. Cái nào tốt hơn? Vì sao?

## Bài 3: Hard-margin từ scratch (chỉ làm khi đã có cvxopt)
Cài `pip install cvxopt`. Implement hard-margin SVM bằng quadratic programming:
1. Thiết lập ma trận P, q, G, h, A, b cho dual problem.
2. Gọi `cvxopt.solvers.qp()`.
3. Lấy support vectors (có $\lambda_i > 10^{-5}$).
4. Tính w, b. Vẽ ranh giới.

*Gợi ý:* $P_{ij} = y_i y_j x_i^T x_j$, $q = -\mathbf{1}$, ràng buộc $\lambda_i \ge 0$ là $G = -I$, $h = 0$. Ràng buộc $\sum \lambda_i y_i = 0$ là $A = y^T$, $b = 0$.

## Bài 4: Imbalanced classes
Sinh dữ liệu mất cân bằng: 950 mẫu lớp 0, 50 mẫu lớp 1. Train SVM. Quan sát: accuracy cao nhưng recall lớp 1 thấp. Thử `class_weight='balanced'` — cải thiện thế nào? Vì sao?

## Bài 5: Probability calibration
SVM `decision_function` cho điểm số, không phải xác suất. Dùng `CalibratedClassifierCV` (Platt scaling) để có xác suất tin cậy. Vẽ ROC curve cho SVM trên Iris (binary: setosa vs not-setosa). So sánh AUC trước và sau calibration.

*Gợi ý:* `from sklearn.calibration import CalibratedClassifierCV`.